# Increment 9 — Dynamic elastic properties and QC

**Poseidon 2 • p2mem 0.9.0 • Tier C: screening, uncalibrated, educational**

This notebook reviews the verified results for all four approved wells. It can
also reproduce them from the private source files when you explicitly select
`regenerate` mode. Review mode does **not** claim to re-read private measurements.

Upload `Poseidon_1D_MEM_Increment_09.zip` to `My Drive/Poseidon_1D_MEM/`
or the root of My Drive, then choose **Runtime → Run all**. You do not need
to extract or run earlier increments. The ZIP contains the locked foundation.
The notebook uses a separate local working folder and leaves older Drive folders intact.

The package already contains its Python implementation and tests; this notebook
does not overwrite source files with `%%writefile`. The full test suite runs in
a separate Python process so older notebook imports cannot select an old package.

## 1. Choose the run mode

Leave these settings unchanged for your first run. For regeneration, set
`MODE = "regenerate"` and `PRIVATE_INPUT_ROOT` to a folder containing
`las/` (the four approved `*_logs.las` files) and `deviation/` (the four approved
`*_dev.txt` files, with spaces in their filenames). File identity is checked by
the existing per-file contracts. Missing files stop regeneration.

For local Jupyter, set `PROJECT_ROOT` to a fresh extraction of this release,
or set `ZIP_PATH` to the release ZIP. Use Python 3.9 or newer.

In [ ]:
MODE = "review"                  # "review" or "regenerate"
ZIP_PATH = ""                    # Optional exact ZIP path
PROJECT_ROOT = ""                # Optional existing verified local extraction
PRIVATE_INPUT_ROOT = ""          # Required only for regeneration
INSTALL_DEPENDENCIES = True


## 2. Locate, extract and verify this release

Every ledger-covered file is checked before project code runs. Extraction uses
a new folder tied to the ZIP content hash. An existing verified folder may be
reused; a damaged folder is reported rather than overwritten. These checks
detect corruption. Compare the ZIP hash with the separately supplied completion
record if you also need to establish the archive's release identity.

In [ ]:
import hashlib
import os
from pathlib import Path, PurePosixPath
import stat
import tempfile
from zipfile import ZipFile

def verify_tree(root):
    ledger = root / "INCREMENT_09_SHA256SUMS.txt"
    if not ledger.is_file():
        raise FileNotFoundError(f"Increment 9 ledger missing in {root}. Use the Increment 9 ZIP.")
    seen = set()
    for line in ledger.read_text(encoding="utf-8").splitlines():
        if not line or line.startswith("#"):
            continue
        digest, relative = line.split("  ", 1)
        p = PurePosixPath(relative)
        if p.is_absolute() or ".." in p.parts or "\\" in relative or ":" in relative or relative in seen:
            raise ValueError("Invalid package ledger path")
        if len(digest) != 64 or any(c not in "0123456789abcdef" for c in digest):
            raise ValueError("Invalid package ledger checksum")
        seen.add(relative)
        target = root / relative
        if not target.is_file() or target.is_symlink() or hashlib.sha256(target.read_bytes()).hexdigest() != digest:
            raise RuntimeError(f"Package file missing or changed: {relative}. Select a fresh extraction.")
    if not seen:
        raise ValueError("Empty package ledger")
    return seen

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

if PROJECT_ROOT:
    project_root = Path(PROJECT_ROOT).expanduser().resolve()
    verify_tree(project_root)
else:
    if ZIP_PATH:
        archive = Path(ZIP_PATH).expanduser().resolve()
    else:
        candidates = [Path("/content/drive/MyDrive/Poseidon_1D_MEM/Poseidon_1D_MEM_Increment_09.zip"),
                      Path("/content/drive/MyDrive/Poseidon_1D_MEM_Increment_09.zip")]
        available = [p for p in candidates if p.is_file()]
        if len(available) != 1:
            raise FileNotFoundError("Set ZIP_PATH to your exact Increment 9 ZIP path; no unique default ZIP was found.")
        archive = available[0]
    archive_hash = hashlib.sha256(archive.read_bytes()).hexdigest()
    print("ZIP SHA-256:", archive_hash)
    parent = Path("/content") if IN_COLAB else Path(tempfile.gettempdir())
    project_root = parent / ("poseidon_inc09_" + archive_hash[:16])
    if not project_root.exists():
        with ZipFile(archive) as z:
            names = set()
            for info in z.infolist():
                name = info.filename
                p = PurePosixPath(name)
                if (p.is_absolute() or ".." in p.parts or "\\" in name or ":" in name
                    or p.as_posix() != name or name in names or info.is_dir()
                    or stat.S_ISLNK(info.external_attr >> 16)):
                    raise ValueError("Unsafe or duplicate ZIP entry: " + name)
                names.add(name)
            if sum(i.file_size for i in z.infolist()) > 200_000_000:
                raise ValueError("Unexpected archive size")
            staging = Path(tempfile.mkdtemp(prefix="poseidon_inc09_extract_", dir=parent))
            z.extractall(staging)
        listed = verify_tree(staging)
        if names != listed | {"INCREMENT_09_SHA256SUMS.txt"}:
            raise RuntimeError("ZIP and ledger inventories differ")
        staging.rename(project_root)
    if project_root.is_symlink():
        raise RuntimeError("Project folder cannot be a symlink")
    verify_tree(project_root)

print("Verified project folder:", project_root)
print("Mode:", MODE)


## 3. Install the package

The installation uses the verified local source. Dependencies may download if
the runtime does not already have them. Set `INSTALL_DEPENDENCIES = False`
only in an environment where NumPy, PyYAML, Matplotlib, pytest and setuptools
are already installed. No source file is rewritten by notebook code.

In [ ]:
import subprocess
import sys

if INSTALL_DEPENDENCIES:
    install = [sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "-q",
               "-e", str(project_root) + "[dev]"]
else:
    install = [sys.executable, "-m", "pip", "install", "--no-index", "--no-deps",
               "--no-build-isolation", "-q", "-e", str(project_root)]
subprocess.run(install, check=True, cwd=project_root)
verify_tree(project_root)
print("Installation completed.")


## 4. Run validation, calculations when selected, and the full tests

This step checks the Increment 8.0.1 foundation and the actual exported records,
then runs the full suite in a disposable copy, because some prior integration
tests regenerate their QC figures. Test counts are read from pytest's XML report rather
than matched to a fragile console string. The 1,632 prior tests must remain present.
Any failure stops the notebook. Review and regeneration are reported separately.

In [ ]:
if MODE not in ("review", "regenerate"):
    raise ValueError("MODE must be review or regenerate")
command = [sys.executable, str(project_root / "scripts/run_increment_09.py"), "--mode", MODE]
if MODE == "regenerate":
    if not PRIVATE_INPUT_ROOT:
        raise ValueError("Set PRIVATE_INPUT_ROOT for regeneration")
    command.extend(["--private-input-root", PRIVATE_INPUT_ROOT])
elif PRIVATE_INPUT_ROOT:
    raise ValueError("Private inputs were set in review mode. Choose regenerate explicitly.")
completed = subprocess.run(command, cwd=project_root, text=True, stdout=subprocess.PIPE,
                           stderr=subprocess.STDOUT, check=False)
print(completed.stdout)
if completed.returncode != 0:
    raise RuntimeError("Increment 9 did not pass. Read the first error above; do not skip this check.")


## 5. Read the measured coverage and dynamic estimates

For a linear isotropic medium, $G=\rho V_s^2$ and
$K=\rho(V_p^2-4V_s^2/3)$. Combining these with the isotropic elastic identities
gives $\nu=(r^2-2)/(2(r^2-1))$ and $E=9KG/(3K+G)$, where $r=V_p/V_s$.
Inputs are m/s and kg/m³; output moduli are GPa.
[MIT wave-speed relations](https://ocw.mit.edu/courses/12-510-introduction-to-seismology-spring-2010/d77d74b5471755994a9dae8a19eddba0_lec1.pdf),
[MIT elastic identities](https://ocw.mit.edu/courses/22-314j-structural-mechanics-in-nuclear-power-technology-fall-2006/137e6469e37e9d347b7b3b69292da2f3_l4_2.pdf).

Poisson's ratio needs both velocities. Shear modulus needs Vs and density.
Bulk and Young's moduli need both velocities and density. All primary results
also require a mapped depth; ratio-dependent results retain the inherited
$\sqrt{2} \le r \le 4$ applicability screen.

**Negative Poisson's ratio is not automatically impossible.** Between
$\sqrt{4/3}$ and $\sqrt{2}$, positive bulk modulus is compatible with negative
Poisson's ratio. These samples retain a separately named diagnostic, while
primary ν/K/E remain withheld under the existing conservative policy.
Ratios at or below $\sqrt{4/3}$ imply non-positive bulk modulus under the
isotropic model. These two categories are counted separately.

In [ ]:
import csv
import json
from IPython.display import display, Markdown, Image

output_root = project_root / "outputs" / (
    "09_dynamic_elasticity_regenerated" if MODE == "regenerate" else "09_dynamic_elasticity")
manifest = json.loads((output_root / "elastic_manifest.json").read_text(encoding="utf-8"))
rows = ["| Well | ν samples | G samples | K/E samples | Median E (GPa) |",
        "|---|---:|---:|---:|---:|"]
for well, item in manifest["wells"].items():
    c = item["counts"]
    median = next(s["median"] for s in item["statistics"] if s["property"] == "E_dynamic_GPa")
    value = "unavailable" if median is None else f"{median:.3f}"
    rows.append(f"| {well.replace('_', ' ')} | {c['nu_valid']:,} | {c['G_valid']:,} | {c['E_valid']:,} | {value} |")
display(Markdown("\n".join(rows)))
diagnostics = ["| Well | Non-positive K regime | Stable negative ν regime |",
               "|---|---:|---:|"]
for well, item in manifest["wells"].items():
    c=item["counts"]
    diagnostics.append(f"| {well.replace('_',' ')} | {c['nonpositive_bulk_ratio']} | {c['negative_poisson_ratio']} |")
display(Markdown("\n".join(diagnostics)))


## 6. Inspect the QC figures

Each figure zooms to the interval with available elastic estimates. Counts use
the entire original LAS sample population. Gaps remain blank, with no lines
joining missing samples. Values outside the screening policy are withheld.

In [ ]:
for well in manifest["wells"]:
    display(Image(filename=str(output_root / f"{well}_elastic_qc.png")))


## 7. Interpret the limitations and save your results

- These are **dynamic isotropic estimates**, not calibrated static moduli or strength.
- No static conversion, strength correlation, horizontal stress or stability calculation is added.
- Sparse Vs and density coverage limits usable results. Coverage counts are not evidence of calibration.
- Measurement uncertainty, anisotropy, tool-quality effects and frequency/strain dependence remain unquantified.
- `K_velocity_condition_number` is an algebraic sensitivity diagnostic at fixed density. It is not a measured uncertainty, confidence interval or calibration.
- Boreas 1's GR exclusion is independent of the velocity/density evidence used here.
- The private sources remain outside the deliverable. Provenance retains their basenames, hashes and contract conversions.

The next cell saves only this increment's derived tables, figures and run record
to a new results ZIP. It never includes the private inputs. In Colab, the results
ZIP is saved into `My Drive/Poseidon_1D_MEM/`. Increment 10 has not been started.

In [ ]:
from datetime import datetime, timezone
from zipfile import ZipFile, ZIP_DEFLATED

save_parent = Path("/content/drive/MyDrive/Poseidon_1D_MEM") if IN_COLAB else project_root / "run_records"
save_parent.mkdir(parents=True, exist_ok=True)
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
results_zip = save_parent / f"Poseidon_Increment_09_{MODE}_results_{stamp}.zip"
with ZipFile(results_zip, "x", compression=ZIP_DEFLATED) as z:
    for file in sorted(output_root.iterdir()):
        z.write(file, "outputs/09_dynamic_elasticity/" + file.name)
    for name in ("increment_09_run.json", "increment_09_pytest.log", "increment_09_pytest.xml"):
        z.write(project_root / "run_records" / name, "run_records/" + name)
print("Saved:", results_zip)
print("Completed mode:", MODE)
